In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/pambifree/02-data-audit/__huggingface_repos__.json
/kaggle/input/notebooks/pambifree/02-data-audit/__notebook__.ipynb
/kaggle/input/notebooks/pambifree/02-data-audit/__output__.json
/kaggle/input/notebooks/pambifree/02-data-audit/__results__.html
/kaggle/input/notebooks/pambifree/02-data-audit/custom.css
/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/test.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/train.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/val.csv


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
!find /kaggle/input -name "*.csv" | head -20

/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/test.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/train.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/val.csv


In [4]:
import time, gc, random
import numpy as np, pandas as pd, torch
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

PUBLIC_DIR = "/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1"    # replace
GOLD_PATH = "/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv"   # replace
train = pd.read_csv(f"{PUBLIC_DIR}/train.csv")
val = pd.read_csv(f"{PUBLIC_DIR}/val.csv")
test = pd.read_csv(f"{PUBLIC_DIR}/test.csv")
gold = pd.read_csv(GOLD_PATH)

SEED = 42
def cap_groups(df, cap=5, seed=SEED):
    return df.sample(frac=1, random_state=seed).groupby("group").head(cap)

def scores(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    legit_ok = ((y_pred == 0) & (y_true == 0)).sum() / max((y_true == 0).sum(), 1)
    return {"precision": p, "recall": r, "f1": f, "accuracy": accuracy_score(y_true, y_pred),
            "false_alarm_rate": 1 - legit_ok, "balanced_acc": (r + legit_ok) / 2}

val_c, test_c = cap_groups(val), cap_groups(test)
print(train.shape, val_c.shape, test.shape, gold.shape)

device: cuda
(7154, 5) (1350, 5) (2066, 5) (326, 7)


**Step 2: The training code**

In [5]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

class TextDS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = list(texts), list(labels)
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i], int(self.labels[i])

def make_loader(tok, texts, labels, batch_size, shuffle, max_length=128, seed=0):
    def collate(batch):
        t, y = zip(*batch)
        enc = tok(list(t), padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        enc["labels"] = torch.tensor(y)
        return enc
    gen = torch.Generator(); gen.manual_seed(seed)
    return torch.utils.data.DataLoader(TextDS(texts, labels), batch_size=batch_size, shuffle=shuffle,
                                       collate_fn=collate, generator=gen if shuffle else None)

@torch.no_grad()
def predict_proba(model, loader):
    model.eval(); out = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(**batch).logits
        out.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu())
    return torch.cat(out).numpy()

def fine_tune(model_path, seed, train_df, val_df, eval_sets, epochs=3, lr=2e-5,
              batch_size=32, max_length=128, warmup_frac=0.1):
    seed_everything(seed)
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2).to(device)
    tr_loader = make_loader(tok, train_df["text"], train_df["is_scam"], batch_size, True, max_length, seed)
    va_loader = make_loader(tok, val_df["text"], val_df["is_scam"], 128, False, max_length)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = epochs * len(tr_loader); warm = max(int(warmup_frac * total), 1)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(total - warm, 1)))
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    best_ba, best_epoch, best_state = -1.0, 0, None
    for ep in range(1, epochs + 1):
        model.train(); running = []
        for batch in tr_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                loss = model(**batch).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            running.append(loss.item())
        vp = predict_proba(model, va_loader)
        ba = scores(val_df["is_scam"], (vp >= 0.5).astype(int))["balanced_acc"]
        print(f"  epoch {ep}: train loss {np.mean(running):.3f} | val balanced acc {ba:.3f}")
        if ba > best_ba:
            best_ba, best_epoch = ba, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    result = {"best_epoch": best_epoch, "val_ba": best_ba, "val": predict_proba(model, va_loader)}
    for name, df_ in eval_sets.items():
        loader = make_loader(tok, df_["text"], np.zeros(len(df_), dtype=int), 128, False, max_length)
        result[name] = predict_proba(model, loader)
    del model, opt, best_state
    gc.collect(); torch.cuda.empty_cache()
    return result

**Step 3. Debug run on CPU**

In [6]:
t0 = time.time()
tiny_train = train.sample(200, random_state=1)
tiny_val = val_c.head(80)
res = fine_tune("l3cube-pune/hing-bert", 42, tiny_train, tiny_val, {"gold": gold.head(40)}, epochs=1)
print("OK | gold probs:", res["gold"][:5].round(2), f"| {time.time() - t0:.0f} s")

config.json:   0%|          | 0.00/716 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: l3cube-pune/hing-bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1: train loss 0.662 | val balanced acc 0.913
OK | gold probs: [0.51 0.52 0.49 0.52 0.51] | 13 s


**Step-4: The full run**

In [7]:
MODELS = {"HingBERT": "l3cube-pune/hing-bert", "XLM-R base": "xlm-roberta-base"}
SEEDS = [42, 43, 44]
os.makedirs("/kaggle/working/probs", exist_ok=True)

run_log = []
for name, path in MODELS.items():
    for seed in SEEDS:
        print(f"\n{name}, seed {seed}")
        t0 = time.time()
        res = fine_tune(path, seed, train, val_c, {"test": test, "gold": gold})
        tag = name.replace(" ", "_")
        np.savez_compressed(f"/kaggle/working/probs/{tag}_seed{seed}.npz",
                            val=res["val"], test=res["test"], gold=res["gold"])
        run_log.append({"model": name, "seed": seed, "best_epoch": res["best_epoch"],
                        "val_ba": round(res["val_ba"], 3), "minutes": round((time.time() - t0) / 60, 1)})
        print(f"  -> best epoch {res['best_epoch']}, {run_log[-1]['minutes']} min")

run_log = pd.DataFrame(run_log)
run_log


HingBERT, seed 42


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: l3cube-pune/hing-bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1: train loss 0.193 | val balanced acc 0.962
  epoch 2: train loss 0.046 | val balanced acc 0.975
  epoch 3: train loss 0.021 | val balanced acc 0.976
  -> best epoch 3, 1.9 min

HingBERT, seed 43


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: l3cube-pune/hing-bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (16 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1: train loss 0.271 | val balanced acc 0.962
  epoch 2: train loss 0.078 | val balanced acc 0.965
  epoch 3: train loss 0.044 | val balanced acc 0.972
  -> best epoch 3, 2.8 min


,model,seed,best_epoch,val_ba,minutes
0,HingBERT,42,3,0.976,1.9
1,HingBERT,43,2,0.973,1.9
2,HingBERT,44,3,0.975,1.9
3,XLM-R base,42,3,0.972,3.0
4,XLM-R base,43,2,0.969,2.8
5,XLM-R base,44,3,0.972,2.8


**Step 5. Results: mean ± spread, next to everything so far**

In [8]:
cols = ["precision", "recall", "f1", "false_alarm_rate", "balanced_acc"]
rows = []
for name in MODELS:
    for seed in SEEDS:
        p = np.load(f"/kaggle/working/probs/{name.replace(' ', '_')}_seed{seed}.npz")
        for set_name, probs, y in [("public test (capped)", p["test"][test_c.index], test_c["is_scam"]),
                                   ("public test (all rows)", p["test"], test["is_scam"]),
                                   ("gold v1", p["gold"], gold["label"])]:
            rows.append({"eval set": set_name, "model": f"{name} (fine-tuned)", "seed": seed,
                         **scores(y, (probs >= 0.5).astype(int))})
ft = pd.DataFrame(rows)

print("Fine-tuned models, mean and std over 3 seeds:")
display(ft.groupby(["eval set", "model"])[cols].agg(["mean", "std"]).round(3))

BASE = "https://raw.githubusercontent.com/Pambifreeman/hinglish-scam-detector/main/reports/"
prev = pd.concat([pd.read_csv(BASE + "baseline_final_results.csv"), pd.read_csv(BASE + "frozen_embedding_results.csv")])
prev = prev[prev["model"] != "Majority class"]
ft_mean = ft.groupby(["eval set", "model"])[cols].mean().round(3).reset_index()
comparison = pd.concat([prev, ft_mean]).set_index(["eval set", "model"]).sort_index()
comparison[cols]


Fine-tuned models, mean and std over 3 seeds:


precision        recall         \
                                                    mean    std   mean    std   
eval set               model                                                    
gold v1                HingBERT (fine-tuned)       0.679  0.003  0.862  0.044   
                       XLM-R base (fine-tuned)     0.669  0.005  0.836  0.014   
public test (all rows) HingBERT (fine-tuned)       0.657  0.091  0.984  0.002   
                       XLM-R base (fine-tuned)     0.612  0.009  0.984  0.005   
public test (capped)   HingBERT (fine-tuned)       0.951  0.004  0.982  0.002   
                       XLM-R base (fine-tuned)     0.949  0.004  0.982  0.005   

                                                   f1        false_alarm_rate  \
                                                 mean    std             mean   
eval set               model                                                    
gold v1                HingBERT (fine-tuned)    0.759  0.017            0.522   
                       XLM-R base (fine-tuned)  0.743  0.007            0.529   
public test (all rows) HingBERT (fine-tuned)    0.786  0.064            0.353   
                       XLM-R base (fine-tuned)  0.754  0.006            0.416   
public test (capped)   HingBERT (fine-tuned)    0.967  0.002            0.056   
                       XLM-R base (fine-tuned)  0.965  0.001            0.059   

                                                      balanced_acc         
                                                  std         mean    std  
eval set               model                                               
gold v1                HingBERT (fine-tuned)    0.028        0.670  0.009  
                       XLM-R base (fine-tuned)  0.015        0.653  0.007  
public test (all rows) HingBERT (fine-tuned)    0.129        0.816  0.065  
                       XLM-R base (fine-tuned)  0.017        0.784  0.007  
public test (capped)   HingBERT (fine-tuned)    0.004        0.963  0.002  
                       XLM-R base (fine-tuned)  0.005        0.961  0.001

precision  recall     f1  \
eval set               model                                                    
gold v1                HingBERT (fine-tuned)             0.679   0.862  0.759   
                       HingBERT (frozen) + LogReg        0.716   0.770  0.742   
                       Naive Bayes (word counts)         0.624   0.978  0.762   
                       TF-IDF chars + LogReg             0.688   0.842  0.757   
                       TF-IDF words + LogReg             0.644   0.891  0.748   
                       XLM-R base (fine-tuned)           0.669   0.836  0.743   
                       XLM-R base (frozen) + LogReg      0.715   0.699  0.707   
public test (all rows) HingBERT (fine-tuned)             0.657   0.984  0.786   
                       HingBERT (frozen) + LogReg        0.782   0.960  0.862   
                       Naive Bayes (word counts)         0.587   0.967  0.730   
                       TF-IDF chars + LogReg             0.598   0.970  0.740   
                       TF-IDF words + LogReg             0.593   0.962  0.734   
                       XLM-R base (fine-tuned)           0.612   0.984  0.754   
                       XLM-R base (frozen) + LogReg      0.783   0.967  0.866   
public test (capped)   HingBERT (fine-tuned)             0.951   0.982  0.967   
                       HingBERT (frozen) + LogReg        0.951   0.955  0.953   
                       Naive Bayes (word counts)         0.911   0.963  0.936   
                       TF-IDF chars + LogReg             0.940   0.966  0.953   
                       TF-IDF words + LogReg             0.932   0.958  0.945   
                       XLM-R base (fine-tuned)           0.949   0.982  0.965   
                       XLM-R base (frozen) + LogReg      0.948   0.963  0.955   

                                                     false_alarm_rate  \
eval set               model                                            
gold v1                HingBERT (fine-tuned)                    0.522   
                       HingBERT (frozen) + LogReg               0.392   
                       Naive Bayes (word counts)                0.755   
                       TF-IDF chars + LogReg                    0.490   
                       TF-IDF words + LogReg                    0.629   
                       XLM-R base (fine-tuned)                  0.529   
                       XLM-R base (frozen) + LogReg             0.357   
public test (all rows) HingBERT (fine-tuned)                    0.353   
                       HingBERT (frozen) + LogReg               0.178   
                       Naive Bayes (word counts)                0.453   
                       TF-IDF chars + LogReg                    0.434   
                       TF-IDF words + LogReg                    0.438   
                       XLM-R base (fine-tuned)                  0.416   
                       XLM-R base (frozen) + LogReg             0.178   
public test (capped)   HingBERT (fine-tuned)                    0.056   
                       HingBERT (frozen) + LogReg               0.055   
                       Naive Bayes (word counts)                0.106   
                       TF-IDF chars + LogReg                    0.069   
                       TF-IDF words + LogReg                    0.078   
                       XLM-R base (fine-tuned)                  0.059   
                       XLM-R base (frozen) + LogReg             0.060   

                                                     balanced_acc  
eval set               model                                       
gold v1                HingBERT (fine-tuned)                0.670  
                       HingBERT (frozen) + LogReg           0.689  
                       Naive Bayes (word counts)            0.611  
                       TF-IDF chars + LogReg                0.676  
                       TF-IDF words + LogReg                0.631  
                       XLM-R base

**Step 6. Gold breakdown**

In [9]:
best_name = run_log.groupby("model")["val_ba"].mean().idxmax()
print("Best by validation balanced accuracy:", best_name)

p = np.load(f"/kaggle/working/probs/{best_name.replace(' ', '_')}_seed42.npz")
g = gold.copy()
g["pred"] = (p["gold"] >= 0.5).astype(int)
g["correct"] = g["pred"] == g["label"]
print(g.groupby("type")["correct"].agg(["mean", "size"]).round(2), "\n")
print(g.groupby("tag")["correct"].agg(["mean", "size"]).round(2).sort_values("mean").head(12))

Best by validation balanced accuracy: HingBERT
       mean  size
type             
legit  0.48   143
promo  0.98    65
scam   0.81   118 

                mean  size
tag                       
has_link        0.05    21
hinglish_alert  0.07    15
alarming        0.15    20
scam_warning    0.20    10
transactional   0.35    20
very_long       0.50     2
hinglish_scam   0.64    25
other_scam      0.69    13
shorthand       0.71    24
extortion_scam  0.80     5
obfuscated      0.88    16
money_chat      0.95    20


**Step 7. Save, commit, switch the GPU off**

In [10]:
ft.to_csv("/kaggle/working/finetune_runs.csv", index=False)
run_log.to_csv("/kaggle/working/finetune_run_log.csv", index=False)